# 01. Jev 기초: 글을 쓰지 않고 판단만 하는 모델

**비법노트 주주총회 · 테디노트**

Jev 가 왜 이렇게 동작하는지(구조, confidence 공식, 학습 방식)는 00번 노트북에 정리했습니다. 이 노트북은 사용법에 집중합니다.

## 배경

에이전트는 한 번 실행되는 동안 작은 판단을 수십 번 내립니다. 어떤 도구를 쓸지, 이 요청이 위험한지, 작업이 끝났는지 같은 것들입니다. 지금까지는 이 판단을 전부 큰 LLM 에게 맡겼습니다. 그러면 판단 하나에 몇 초가 걸리고, 답이 문장으로 나오기 때문에 다시 파싱해야 하며, 없는 도구 이름을 지어내는 일도 생깁니다.

Jev 는 TypeSafe 가 2026년 9월에 공개한 모델로, 이런 판단만 전담합니다. 대니얼 카너먼의 책 《생각에 관한 생각》에 나오는 빠르고 직관적인 사고 방식인 System 1 에서 이름을 따 "System One 모델"이라고 부릅니다.

## 직관

객관식 시험을 떠올리면 됩니다. LLM 은 서술형 답안을 쓰는 학생이고, Jev 는 option 중 하나에 표시하고 "이 정도로 확신한다"는 숫자까지 적는 학생입니다. option 에 없는 답은 낼 수 없습니다.

| 항목 | 값 |
|---|---|
| 모델 ID | `typesafe/jev-1.13` |
| 엔드포인트 | `POST https://openrouter.ai/api/alpha/decisions` (채팅 API 가 아님) |
| 입력 | `state`(문자열 또는 JSON) + `questions`(타입이 정해진 질문 묶음) |
| 출력 | 질문마다 답과 probability. 텍스트는 생성하지 않음 |
| 가격 | 입력 100만 토큰당 $0.042, 출력 무료 |
| 컨텍스트 | 32,000 토큰 |
| 지연 | TypeSafe 발표 기준 70~500ms |

질문 타입은 세 가지입니다.

- Choice: 여러 option 중 하나
- Noul: 예/아니오. "예"일 probability 하나를 돌려줌
- Score: 순서가 있는 척도 위의 위치

## 준비

프로젝트 루트의 `.env` 에 `OPENROUTER_API_KEY` 가 있어야 합니다. Jev 는 선불 크레딧이 있는 계정에서만 호출됩니다.

In [1]:
import sys
sys.path.insert(0, "..")  # 프로젝트 루트의 jev_agent 패키지를 불러오기 위함

from dotenv import load_dotenv
load_dotenv("../.env")

from jev_agent.jev import JevClient, choice, noul, score

jev = JevClient()

## 가장 단순한 호출

`JevClient` 가 하는 일은 HTTP POST 한 번이 전부입니다. 먼저 래퍼 없이 직접 호출해서 요청과 응답의 모양을 확인합니다.

In [2]:
import os, json, httpx

response = httpx.post(
    "https://openrouter.ai/api/alpha/decisions",
    headers={"Authorization": f"Bearer {os.environ['OPENROUTER_API_KEY']}"},
    json={
        "model": "typesafe/jev-1.13",
        "state": "어제 주문한 운동화가 아직 안 왔어요. 어디쯤인가요?",
        "questions": {
            "is_complaint": {
                "type": "noul",
                "instructions": "고객이 불만을 표현하고 있는가?",
            }
        },
    },
    timeout=15,
)
print(json.dumps(response.json(), ensure_ascii=False, indent=2))

{
  "model": "typesafe/jev-1.13-20260917",
  "answers": {
    "is_complaint": {
      "type": "noul",
      "noul": 0.8
    }
  },
  "usage": {
    "input_tokens": 306,
    "output_tokens": 23,
    "cost": 1.2852e-05
  },
  "id": "gen-dec-1791020616-oriIVrQ3bBUNYE2bGkDZ",
  "provider": "TypeSafe"
}


응답에서 볼 것은 `answers` 와 `usage` 입니다. `usage.cost` 는 달러 단위 실제 과금액입니다.

## Choice: option 중 하나 고르기

`criteria` 에 `{라벨: 설명}` 을 넣습니다. 답은 반드시 이 라벨 중 하나입니다.

In [3]:
ticket = "결제 버튼을 누르면 화면이 하얗게 변해요. 브라우저 두 개에서 다 그래요."

result = jev.decide(
    ticket,
    {
        "team": choice(
            "이 문의는 어느 팀이 맡아야 하는가?",
            {
                "payments": "결제, 청구, 환불 문제",
                "frontend": "화면 표시, 레이아웃, 브라우저 호환 문제",
                "account": "로그인, 권한, 프로필 문제",
            },
        )
    },
)
result["team"]

{'type': 'choice',
 'choice': 'frontend',
 'probabilities': {'account': 0, 'frontend': 0.83, 'payments': 0.17},
 'confidence': 0.74}

`choice` 는 고른 라벨, `probabilities` 는 option 별 probability, `confidence` 는 이 선택을 얼마나 믿을 수 있는지입니다. 03번 노트북에서 이 `confidence` 로 분기를 만듭니다.

## Noul: 예/아니오

답은 0과 1 사이 숫자 하나입니다. 기준을 `true`/`false` 설명으로 적어 주면 판단이 더 안정됩니다.

In [4]:
result = jev.decide(
    ticket,
    {
        "is_bug": noul(
            "고객이 소프트웨어 결함을 신고하고 있는가?",
            true="제품이 고장났거나 예상과 다르게 동작한다고 설명한다.",
            false="사용법을 묻거나 기능을 요청한다.",
        )
    },
)
result["is_bug"]

{'type': 'noul', 'noul': 0.97}

## Score: 척도 위의 위치

단계를 낮은 쪽부터 리스트로 적습니다. `score` 는 probability 로 가중한 위치이고 `legend` 는 번호와 단계 설명의 대응표입니다.

In [5]:
result = jev.decide(
    ticket,
    {
        "urgency": score(
            "이 문의는 얼마나 급한가?",
            ["다음 배포 때 고쳐도 된다", "이번 주 안에 고쳐야 한다", "지금 매출이 막히고 있다"],
        )
    },
)
result["urgency"]

{'type': 'score',
 'score': 1.87,
 'legend': {'0': '다음 배포 때 고쳐도 된다',
  '1': '이번 주 안에 고쳐야 한다',
  '2': '지금 매출이 막히고 있다'},
 'probabilities': {'0': 0, '1': 0.12, '2': 0.88},
 'confidence': 0.81}

## 여러 질문을 한 번에

질문 묶음은 한 번의 호출 안에서 병렬로 평가됩니다. 질문을 세 개 보내도 왕복은 한 번입니다.

In [6]:
result = jev.decide(
    {"customer_tier": "enterprise", "ticket": ticket},
    {
        "team": choice("어느 팀이 맡아야 하는가?", {
            "payments": "결제, 청구, 환불 문제",
            "frontend": "화면 표시, 브라우저 호환 문제",
            "account": "로그인, 권한 문제",
        }),
        "is_bug": noul("소프트웨어 결함 신고인가?"),
        "urgency": score("얼마나 급한가?", ["나중에", "이번 주", "지금 당장"]),
    },
)

for name, answer in result.answers.items():
    print(name, "→", answer)
print(f"\n지연 {result.latency_ms:.0f}ms · 입력 토큰 {result.usage.get('input_tokens')} · 비용 ${result.cost:.8f}")

team → {'type': 'choice', 'choice': 'frontend', 'probabilities': {'account': 0, 'frontend': 0.76, 'payments': 0.24}, 'confidence': 0.64}
is_bug → {'type': 'noul', 'noul': 0.93}
urgency → {'type': 'score', 'score': 1.97, 'legend': {'0': '나중에', '1': '이번 주', '2': '지금 당장'}, 'probabilities': {'0': 0, '1': 0.02, '2': 0.98}, 'confidence': 0.96}

지연 345ms · 입력 토큰 459 · 비용 $0.00001928


## 정리

- Jev 는 채팅 모델이 아니라서 `ChatOpenAI` 로 부를 수 없습니다. Decisions 엔드포인트에 `state` 와 `questions` 를 보냅니다.
- 답은 우리가 준 option 안에서만 나오고 probability 가 함께 옵니다. 문장을 파싱할 일이 없습니다.
- 문장을 쓰거나 도구 인자를 채우는 일은 하지 못합니다. 그 부분은 여전히 LLM 이 맡습니다.

다음 노트북에서는 이 Choice 를 에이전트의 도구 선택에 씁니다.